# Stellarator coils

The coils of a stellarator from Proxima Fusion's [CoilStellaration](https://huggingface.co/datasets/proxima-fusion/coilstellaration) dataset, with the plasma boundary they were optimized for from [ConStellaration](https://huggingface.co/datasets/proxima-fusion/constellaration) (both MIT-licensed). `freeofcharge.coils` gives their vacuum field (Biot–Savart), the field error on the boundary, its agreement with a vacuum equilibrium inside the plasma, and the coils as obstacles for the beam: which tangential injection lines reach the plasma between them.

Reading a coilset needs plasma-plots, the optional extra `pip install freeofcharge[coils]`.

## The coils and their field

One instance of the dataset's benchmark (3 field periods, 5 coils per half period, its DESC-optimized coilset), the copy kept with freeofcharge's tests. `CoilSet.from_coilset` expands the stored coils over the stellarator symmetry and the field periods (through `plasma_plots.from_coilset`) into closed polylines; `field_at` sums the exact field of their straight segments. The dataset's boundaries are scaled to a major radius of 1 m.

In [ ]:
import json
from pathlib import Path

import numpy as np
import plasma_plots as pp
import pyvista as pv
import xarray as xr

import freeofcharge as foc
from freeofcharge.coils import PlasmaBoundary, injection_access, normal_field_error

pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

data = Path("../src/freeofcharge/tests/data/coilstellaration")
coilset = json.loads((data / "coilstellaration_coilset.json").read_text())
boundary = json.loads((data / "coilstellaration_boundary.json").read_text())
metrics = json.loads((data / "coilstellaration_sample.json").read_text())

coils = foc.CoilSet.from_coilset(coilset, n_points=128)
print(
    f"{coils.n_coils} coils, {coils.nfp} field periods, lengths {coils.lengths.min():.2f}-{coils.lengths.max():.2f} m"
)

The field error $|B_n|/|B|$ on the boundary, from the coils alone: how far the plasma surface is from a flux surface of the vacuum field. Its mean is the dataset's `normalized_field_error/mean`; ours uses polylines of 128 points per coil and central differences for the normal, DESC's its own quadrature.

In [ ]:
surface = pp.from_boundary(
    boundary, theta=64, zeta=np.linspace(0.0, 2 * np.pi, 48 * coils.nfp, endpoint=False)
)
error = normal_field_error(coils, surface)
print(
    f"mean |B_n|/|B| = {100 * float(error.mean()):.3f} %, "
    f"dataset {100 * metrics['desc_metrics/normalized_field_error/mean']:.3f} %"
)
error.attrs.update(long_name=r"$|B_n| / |B|$", units="1")
error.plasma.plot.slice(x="zeta", y="theta")

Outside the plasma the coils give the field an equilibrium code does not: $|B|$ along the major radius at $\varphi = 0$, $Z = 0$, from inside the plasma out past the coils. `foc.CoilField` combines the two regions for the orbits and wall loads: an equilibrium's field inside its last closed surface, the coils' outside.

In [ ]:
R = np.linspace(0.6, 1.6, 401)
line = np.stack([R, 0 * R, 0 * R], -1)
B = np.linalg.norm(coils.field_at(line), axis=-1)
plasma = PlasmaBoundary.from_boundary(boundary)
inside = plasma.inside_at(line)
print(f"plasma between R = {R[inside].min():.3f} and {R[inside].max():.3f} m")
xr.DataArray(
    B / B[inside].mean(),
    dims="R",
    coords={"R": ("R", R, {"label": "$R$", "units": "m"})},
    attrs={"long_name": r"$|B| / \langle |B| \rangle_\mathrm{plasma}$"},
).plasma.plot.lineout(x="R")

## Inside the plasma: the vacuum equilibrium

At $\beta = 0$ and without a plasma current the field inside the boundary is a vacuum field as well, so there the coils' field and a fixed-boundary equilibrium should agree, up to the coils' field error. `VMECEquilibrium.from_boundary` runs [VMEC++](https://github.com/proximafusion/vmecpp) (the optional extra `pip install freeofcharge[vmec]`) on the boundary with zero pressure and zero current, at a low resolution that takes about a second. Then the coil currents are scaled to the equilibrium's field strength (least squares over random points in the plasma, `CoilField(current_scale=...)`), and the two fields are compared at those points.

In [ ]:
from freeofcharge.vmec_equilibrium import VMECEquilibrium

vacuum = VMECEquilibrium.from_boundary(boundary, mpol=8, ntor=7)
rng = np.random.default_rng(0)
lower, upper = vacuum.bounds
candidates = lower + (upper - lower) * rng.random((4000, 3))
points = candidates[vacuum.inside_at(candidates)]
B_vmec = np.asarray(vacuum.field_at(points))
B_coils = np.asarray(coils.field_at(points))
scale = float(np.sum(B_vmec * B_coils) / np.sum(B_coils * B_coils))
combined = foc.CoilField(vacuum, coils, current_scale=scale)
difference = np.asarray(combined.jump(points))
print(f"{len(points)} points in the plasma, coil currents scaled by {scale:.4f}")
print(
    f"|B_coils - B_VMEC| / |B_VMEC|: mean {100 * difference.mean():.2f} %, "
    f"max {100 * difference.max():.2f} %"
)

The difference is at the level of the coils' field error on the boundary (0.29 % above): about 0.2 % in the core ($\rho < 0.5$), larger towards the boundary, where the coils' field error is and VMEC's half-mesh field is extrapolated. With VMEC at only the boundary's own Fourier modes (`mpol=5, ntor=4`) it is about 1 %, so the boundary alone does not resolve the equilibrium. Along the major radius at $\varphi = 0$, $Z = 0$, VMEC's $|B|$ and the coils' (dashed) lie on top of each other:

In [ ]:
in_plasma = np.asarray(vacuum.inside_at(line))
coords = {"R": ("R", R[in_plasma], {"label": "$R$", "units": "m"})}
strength = xr.DataArray(
    np.linalg.norm(vacuum.field_at(line[in_plasma]), axis=-1),
    dims="R",
    coords=coords,
    attrs={"long_name": "$|B|$", "units": "T"},
)
coil_strength = xr.DataArray(
    np.linalg.norm(combined.coil_field_at(line[in_plasma]), axis=-1),
    dims="R",
    coords=coords,
)
strength.plasma.plot.lineout(x="R", label="VMEC++", reference={"coils": coil_strength})

## Injection between the coils

A tangential beam has to pass the coils on its way in. Each coil is a tube around its filament (the winding pack, here 5 cm for a 1 m device), the beam a cylinder of 5 cm radius. `injection_access` follows horizontal axes tangent to $R = R_\mathrm{tan}$ at the toroidal angle $\varphi$, from 2.5 m before the tangency point to where they enter the plasma (`PlasmaBoundary`, the boundary's inside; an equilibrium works the same way), and records the clearance: the distance to the nearest filament minus the tube radius. A line is admissible if the clearance exceeds the beam radius.

In [ ]:
access = injection_access(
    coils,
    plasma,
    tangency_radii=np.linspace(0.6, 1.1, 21),
    toroidal_angles=np.linspace(0.0, 2 * np.pi / coils.nfp, 48, endpoint=False),
    start_distance=2.5,
    coil_radius=0.05,
    beam_radius=0.05,
)
share = float(access.admissible.mean())
print(f"admissible: {100 * share:.0f} % of the lines over one field period")
print(
    "admissible lines per tangency radius:",
    access.admissible.sum("toroidal_angle").isel(vertical_angle=0).values,
)
clearance = access.clearance.isel(vertical_angle=0)
clearance.attrs.update(long_name="clearance", units="m")
clearance.coords["tangency_radius"].attrs.update(label=r"$R_\mathrm{tan}$")
clearance.coords["toroidal_angle"].attrs.update(label=r"$\varphi$")
clearance.plasma.plot.slice(x="toroidal_angle", y="tangency_radius", symmetric=True)

Positive clearance means the axis passes the coils with room to spare. At this coil spacing the windows are few and narrow: at each tangency radius only one to three of the 48 toroidal positions let the axis thread between the coils, fewer towards and beyond the magnetic axis ($R \approx 1$ m), where the axis runs longer alongside the outer coil legs. Below, the coils around the boundary and the admissible axes with the largest clearance, up to the plasma. Click the picture for the interactive scene (vtk.js).

In [ ]:
ok = access.where(access.admissible).clearance.isel(vertical_angle=0)
best = ok.stack(line=("tangency_radius", "toroidal_angle")).dropna("line")
best = best.sortby(best, ascending=False).isel(line=slice(0, 4))
plotter = pp.from_coilset(coilset).plasma.plot.coils_3d(
    surface=pp.from_boundary(boundary), surface_opacity=0.6
)
for r_tan, phi in best.line.values:
    direction = np.array([-np.sin(phi), np.cos(phi), 0.0])
    start = np.array([r_tan * np.cos(phi), r_tan * np.sin(phi), 0.0]) - 2.5 * direction
    length = float(
        access.plasma_distance.sel(tangency_radius=r_tan, toroidal_angle=phi).item()
    )
    plotter.add_mesh(
        pv.Line(start, start + length * direction), color="black", line_width=3
    )
plotter.camera_position = "iso"
pp.scene(
    plotter,
    alt="Stellarator coils around the plasma boundary, with beam axes reaching the plasma between them",
    title="Beam axes between the coils",
)

Across the dataset, the same scan over thousands of configurations and coil-spacing requirements says how much room for neutral beams each kind of coilset leaves; that study, and the beam's markers (`foc.coils.blocked_fraction`, the share of a beam that hits a coil before the plasma), follow in the stellarator plan.